<a href="https://colab.research.google.com/github/mostafadentist/healthcare-data-analytics/blob/main/meals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import openai
import pandas as pd
import os
import time
import requests
import re

# Create the meals folder if it doesn't exist
os.makedirs('meals', exist_ok=True)

# Your Poe API key
API_KEY = ""

# Initialize the OpenAI client with Poe API
client = openai.OpenAI(
    api_key=API_KEY,
    base_url="https://api.poe.com/v1",
)

# Define the meals data
meals_data = {
    'Meal Name': [
        # Breakfast items (Brk 55-80)
        'Quinoa beans with grilled vegetables and marinated tofu',
        'Avocado Toast with Boiled Eggs',
        'Smoothie bowl with fruits and vegetables',
        'Red lentil soup',
        'Rice pies with vegetables',
        'Greek yogurt with dried fruits and nuts',
        'Semolina dessert with fruits and nuts',
        'Greek yogurt parfait with shea and fruits',
        'Refreshing green juice with almonds',
        'Quinoa ice cream with almonds and apples',
        'Healthy omelette with spinach and mushrooms',
        'Quick Snack: Rice biscuits with peanut butter and banana',
        'Rice flour porridge with fruits',
        'Lentil salad with fresh vegetables',
        'Chia seeds with coconut milk and berries',
        'Mashed potatoes with boiled eggs',
        'Refreshing green smoothie',
        'Sponge cake with apple and rice flour',
        'Egg baskets with avocado',
        'Quinoa salad with vegetables',
        'Chia seed pudding',
        'Carrot and apple juice',
        'Salmon Burger with Grilled Vegetables',
        'Egg omelette with vegetables',
        'Rice biscuit snack and jam',
        'Sweet potato soufflé',
        # Lunch items (Lunch 21-41)
        'Tofu salad with vegetables and quinoa',
        'Oats with fruits and nuts',
        'Boiled eggs with vegetables on toast',
        'Grilled salmon with quinoa salad',
        'Chicken salad with fruits and nuts',
        'Pasta with tomato and vegetable sauce',
        'Scrambled eggs with onions and peppers',
        'Tuna sandwich with avocado and vegetables',
        'Grilled chicken salad with vegetables and quinoa',
        'Pasta with pesto sauce and chicken',
        'Salmon with grilled vegetables',
        'Refreshing chickpeas and spinach salad',
        'Warm and nutritious oat soup',
        'Delicious chicken and vegetable pie',
        'Assorted cereal salad rich in fiber',
        'Healthy vegetable and mushroom tart',
        'Roasted Chicken with Sweet Potato',
        'Lentil and vegetable curry',
        'Potato and spinach pancakes',
        'Hummus and tomato salad',
        'Cabbage and carrot soup',
    ],
    'Meal Type': (
        ['Breakfast'] * 26 +
        ['Lunch'] * 21
    ),
    'Number': (
        list(range(55, 81)) +
        list(range(21, 42))
    )
}

# Create DataFrame
df = pd.DataFrame(meals_data)

# Function to create 4K hypercinematic prompts
def create_image_prompt(meal_name, meal_type):
    base_prompt = f"4K hypercinematic food photography of {meal_name}, "

    style_elements = [
        "ultra-detailed close-up shot",
        "professional food styling",
        "natural lighting with soft shadows",
        "shallow depth of field",
        "vibrant colors",
        "appetizing presentation on elegant dishware",
        "rustic wooden table background",
        "fresh ingredients visible",
        "steam rising if hot dish",
        "garnished beautifully",
        "commercial quality",
        "high resolution",
        "photorealistic"
    ]

    full_prompt = base_prompt + ", ".join(style_elements)
    return full_prompt

# Add prompts to DataFrame
df['Image Prompt'] = df.apply(lambda row: create_image_prompt(row['Meal Name'], row['Meal Type']), axis=1)

# Function to extract URL from response
def extract_image_url(response_text):
    # Look for URLs in the response
    url_pattern = r'https://[^\s)\]"]+'
    urls = re.findall(url_pattern, response_text)

    # Filter for image URLs (typically from pfstt.cf2.poecdn.net)
    image_urls = [url for url in urls if 'poecdn.net' in url or 'image' in url.lower()]

    if image_urls:
        return image_urls[0]
    return None

# Function to generate and save image
def generate_and_save_image(prompt, meal_type, number, meal_name):
    try:
        print(f"Generating image for: {meal_name} ({meal_type} {number})...")

        # Generate image using Poe API with FLUX-schnell model
        response = client.chat.completions.create(
            model="FLUX-schnell",
            messages=[{"role": "user", "content": prompt}],
        )

        # Get the response content
        image_content = response.choices[0].message.content

        # Create filename based on meal type
        if meal_type == 'Breakfast':
            file_path = f"meals/Brk_{number}.png"
        else:
            file_path = f"meals/Lunch_{number}.png"

        # Extract image URL from response
        image_url = extract_image_url(image_content)

        if image_url:
            print(f"  Found image URL: {image_url[:60]}...")

            # Download the image
            img_response = requests.get(image_url, timeout=30)

            if img_response.status_code == 200:
                # Save the image
                with open(file_path, 'wb') as f:
                    f.write(img_response.content)
                print(f"✓ Saved: {file_path}")
                return True
            else:
                print(f"✗ Failed to download image. Status code: {img_response.status_code}")
                return False
        else:
            print(f"✗ No image URL found in response")
            print(f"  Response: {image_content[:200]}")
            return False

    except Exception as e:
        print(f"✗ Error generating {meal_type} {number}: {str(e)}")
        return False
    finally:
        # Add delay to avoid rate limiting
        time.sleep(2)

# Save the DataFrame with prompts to CSV for reference
df.to_csv('meals/meal_prompts.csv', index=False)
print("Saved meal prompts to meals/meal_prompts.csv")
print(f"\nTotal meals to generate: {len(df)}")
print(f"Breakfast items: {len(df[df['Meal Type'] == 'Breakfast'])}")
print(f"Lunch items: {len(df[df['Meal Type'] == 'Lunch'])}")
print("\nStarting image generation...\n")

# Generate images for all meals
success_count = 0
failed_meals = []

for index, row in df.iterrows():
    if generate_and_save_image(
        row['Image Prompt'],
        row['Meal Type'],
        row['Number'],
        row['Meal Name']
    ):
        success_count += 1
    else:
        failed_meals.append(f"{row['Meal Type']} {row['Number']}: {row['Meal Name']}")

print(f"\n{'='*60}")
print(f"Generation complete!")
print(f"Successfully generated: {success_count}/{len(df)} images")
print(f"Failed: {len(failed_meals)}")
if failed_meals:
    print("\nFailed meals:")
    for meal in failed_meals[:10]:  # Show first 10 failures
        print(f"  - {meal}")
    if len(failed_meals) > 10:
        print(f"  ... and {len(failed_meals) - 10} more")
print(f"\nImages saved in: ./meals/")
print(f"{'='*60}")

Saved meal prompts to meals/meal_prompts.csv

Total meals to generate: 47
Breakfast items: 26
Lunch items: 21

Starting image generation...

Generating image for: Quinoa beans with grilled vegetables and marinated tofu (Breakfast 55)...
  Found image URL: https://pfst.cf2.poecdn.net/base/image/09cab0ad6c813adcc0281...
✓ Saved: meals/Brk_55.png
Generating image for: Avocado Toast with Boiled Eggs (Breakfast 56)...
  Found image URL: https://pfst.cf2.poecdn.net/base/image/edaf258ac6b28653fe38c...
✓ Saved: meals/Brk_56.png
Generating image for: Smoothie bowl with fruits and vegetables (Breakfast 57)...
  Found image URL: https://pfst.cf2.poecdn.net/base/image/b7f4b313c0e7a0dae30b1...
✓ Saved: meals/Brk_57.png
Generating image for: Red lentil soup (Breakfast 58)...
  Found image URL: https://pfst.cf2.poecdn.net/base/image/be06951b25d57c2c2de6d...
✓ Saved: meals/Brk_58.png
Generating image for: Rice pies with vegetables (Breakfast 59)...
  Found image URL: https://pfst.cf2.poecdn.net/base/i

In [6]:
import shutil
import os

# Copy meals folder to Google Drive
source_folder = '/content/meals'
drive_destination = '/content/drive/MyDrive/meals'

try:
    # Check if source folder exists
    if os.path.exists(source_folder):
        # Remove destination folder if it already exists
        if os.path.exists(drive_destination):
            shutil.rmtree(drive_destination)
            print(f"Removed existing folder: {drive_destination}")

        # Copy the entire folder to Google Drive
        shutil.copytree(source_folder, drive_destination)
        print(f"✓ Successfully copied folder to Google Drive!")
        print(f"  Source: {source_folder}")
        print(f"  Destination: {drive_destination}")

        # Count files
        file_count = len([f for f in os.listdir(drive_destination) if os.path.isfile(os.path.join(drive_destination, f))])
        print(f"  Total files copied: {file_count}")
    else:
        print(f"✗ Source folder not found: {source_folder}")

except Exception as e:
    print(f"✗ Error copying folder: {str(e)}")

✓ Successfully copied folder to Google Drive!
  Source: /content/meals
  Destination: /content/drive/MyDrive/meals
  Total files copied: 48


In [7]:
import shutil
import os
from google.colab import files

# Create zip file of meals folder
source_folder = '/content/meals'
zip_filename = 'meals_images'

try:
    # Check if source folder exists
    if os.path.exists(source_folder):
        print("Creating zip file...")

        # Create zip file (without .zip extension, shutil adds it)
        zip_path = shutil.make_archive(zip_filename, 'zip', source_folder)

        print(f"✓ Zip file created: {zip_path}")

        # Get file size
        file_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
        print(f"  File size: {file_size_mb:.2f} MB")

        # Download the zip file
        print("Downloading zip file...")
        files.download(zip_path)
        print("✓ Download initiated!")

    else:
        print(f"✗ Source folder not found: {source_folder}")

except Exception as e:
    print(f"✗ Error creating zip: {str(e)}")

Creating zip file...
✓ Zip file created: /content/meals_images.zip
  File size: 10.55 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download initiated!
